# Response Hyperalignment (RHA)

In general, RHA consists of three main steps: constructing a common space, estimating subject-specific transformation matrices, and performing alignment.

In this section, we demonstrate these steps using the CIFTI demo data preprocessed in the previous sections.


### 2.1 Constructing the Common Representational Space

In [ ]:
# --------------------------------------------------
# Surface-level RHA: common space construction
# --------------------------------------------------

# Basic path settings
work_dir = '/data/disk0/gaoyixin/clean_program/HAkit_demo/tutorial_data/tutorial_2/ha_workdir_cifti'
sub_list = ['sub-rid000005', 'sub-rid000014', 'sub-rid000029']
json_path = os.path.join(work_dir, 'logs', 'prep_log.json')


# --------------------------------------------------
# Define surface searchlights
#
# Searchlights can be defined in multiple ways:
# - Using nb.sls to generate vertex-centered, overlapping searchlights
# - Using atlas-based parcels (e.g. Schaefer400), where each parcel
#   can be treated as a non-overlapping searchlight
#
# In this tutorial, we recommend using nb.sls on the onavg-ico32 surface,
# which provides a convenient and consistent searchlight definition.
#
# nb.sls generates searchlights on the full surface (including the medial wall).
# Therefore, we further update the searchlight and distance lists by removing
# medial-wall vertices according to the data-specific mask.
# --------------------------------------------------

radius = 20

# Generate initial searchlights and distance lists (including medial wall)
tmp = {
    lr: nb.sls(
        lr,
        radius,
        'onavg-ico32',
        mask=False,
        return_dists=True
    )
    for lr in 'lr'
}

# Separate searchlight indices and distance matrices
sls_lr   = {lr: tmp[lr][0] for lr in tmp}
dists_lr = {lr: tmp[lr][1] for lr in tmp}


# --------------------------------------------------
# Update searchlights by removing medial-wall vertices
#
# medial_wall is a dict of boolean arrays:
# {'l': [...], 'r': [...]}
# where True indicates vertices to be removed.
#
# The updated searchlights and distance lists will match
# the actual response data shape used for RHA.
# --------------------------------------------------
medial_wall = joblib.load(
    '/data/disk0/gaoyixin/clean_program/HAkit_demo/tutorial_data/tutorial_2/cft_onavg32_medial.pkl'
)

sls_surf, dists_surf = gensls.sls_update(
    sls_lr,
    dists_lr,
    medial_wall,
    'lr'
)


# --------------------------------------------------
# Format of searchlight list (for reference)
#
# sls_surf is a dict like:
# {'l': [sl_1, sl_2, ...], 'r': [sl_1, sl_2, ...]}
#
# Each sl_i is a NumPy array containing vertex indices
# within a single searchlight.
# --------------------------------------------------


# --------------------------------------------------
# Construct the common space using searchlight-based RHA
#
# This step estimates local common spaces within each searchlight
# and combines them into a whole-brain common space.
# --------------------------------------------------
ha.cspace_sls(
    work_dir,                               # output directory
    sls=sls_surf,                           # updated searchlight list
    dists=dists_surf,                       # corresponding distance list
    radius=radius,                          # searchlight radius
    sub_list=sub_list,                      # subject list
    task_name='rhaPRs',                     # name of the common space
    cspace_kind='procrustes',               # common space method: 'procrustes' or 'pca'
    common_topography=False,                # required when using PCA-based common space
    weight=True,                            # distance-weighted combination of searchlights
    demean=True,                            # whether to demean data before SVD
    start_sub=0,                            # reference subject for Procrustes ('mean' or index)
    chunk_size=2000,                        # number of searchlights processed per chunk
    njobs=5,                                # number of parallel jobs
    step='resample',                        # folder hierarchy: preprocessing step
    modality='response',                    # modality name
    structure_name=['hemi-L', 'hemi-R'],    # surface structures to use
    dtype=np.float32,                       # output data type
    scope='sls',                            # label for output naming
    json_path=json_path,                    # JSON file to record processing information
    overwrite=False,                        # whether to overwrite existing results
    log_num='raiders_01',                   # log file identifier
    run='01'                                # file filter (can also be passed via file_filters)
)



>>> Calculating common space for hemi-L


100%|██████████████████████████████| 5/5 [00:39<00:00,  7.85s/it]


Common space calculation finished for hemi-L

>>> Calculating common space for hemi-R


100%|██████████████████████████████| 5/5 [00:35<00:00,  7.16s/it]

Common space calculation finished for hemi-R

All common space calculation completed successfully!


In [ ]:
# --------------------------------------------------
# Volume-level RHA: common space construction
# --------------------------------------------------
# Path settings are the same as the surface-based example above


# --------------------------------------------------
# Define subcortical volume searchlights
#
# The volume searchlights are generated in two steps:
# (1) Extract a dense voxel mask for each subcortical structure
#     from a representative CIFTI file
# (2) Generate voxel-centered searchlights within each structure
#     based on the extracted mask
#
# In this example, searchlights are generated separately for:
# - left subcortical volume
# - right subcortical volume
# - brain stem
# --------------------------------------------------

raw_datapath = '/data/disk0/gaoyixin/clean_program/HAkit_demo/tutorial_data/tutorial_2/raw_data/cifti/'

# Extract dense subcortical voxel masks from a reference CIFTI file
mask_dense_vol = {
    lr: gensls.dsample.downsample_cifti_subcortical(
        cifti_file=os.path.join(
            raw_datapath,
            'sub-rid000005_ses-raiders_run-01_TS.dtseries.nii'
        ),
        N=None,
        mask3d_out=None,
        sls_type='seed',
        voxel_sizes=None,
        drop_all_zero_columns=True,
        whole_mask=False,
        hemi_mask=True
    )[lr]
    for lr in ['l', 'r', 'brain_stem']
}


# --------------------------------------------------
# Generate volume searchlights for each subcortical structure
#
# Each searchlight is defined as a local voxel neighborhood
# within the corresponding dense mask.
# --------------------------------------------------
sls_info_vol = {
    lr: gensls.generate_searchlight_vol(
        mask_dense=mask_dense_vol[lr],
        mask_center=None,
        radius=4,
        threshold=0.2,
        njobs=10,
        verbose=5
    )
    for lr in ['l', 'r', 'brain_stem']
}


# --------------------------------------------------
# Organize searchlight indices and distance matrices
#
# The keys of sls_vol and dists_vol must correspond exactly
# to the structure names used in cspace_sls.
# --------------------------------------------------
sls_vol = {
    f"subcortical-{lr}": sls_info_vol[lr]['sls_idx']
    for lr in ['l', 'r', 'brain_stem']
}

dists_vol = {
    f"subcortical-{lr}": sls_info_vol[lr]['dists']
    for lr in ['l', 'r', 'brain_stem']
}


# --------------------------------------------------
# Construct the volume-level common space using RHA
#
# The function cspace_sls is used in the same way as in
# the surface-based example, with volume searchlights
# provided instead of surface searchlights.
# --------------------------------------------------
ha.cspace_sls(
    work_dir,
    sls=sls_vol,
    dists=dists_vol,
    radius=4,
    sub_list=sub_list,
    task_name='rhaPRv',                 # use a different name from the surface part
    cspace_kind='procrustes',
    common_topography=False,
    weight=True,
    demean=True,
    start_sub=0,
    chunk_size=2000,
    njobs=5,
    step='resample',
    modality='response',
    structure_name=[
        'volume-subcortical-L',
        'volume-subcortical-R',
        'volume-subcortical-BRAIN_STEM'
    ],
    dtype=np.float32,
    scope='sls',
    json_path=json_path,
    overwrite=False,
    log_num='raiders_02',
    run='01'
)


### 2.2 Estimating Subject-Specific Transformation Matrices: two workflows

#### 1) Global common space–based workflow (`xform_sls_con*`)

The functions **`xform_sls_con`** and **`xform_sls_con_mega`** require a **complete (global) common space** to be constructed first.

**Workflow:**
1. Combine searchlight-level common spaces into a **global common space**.
2. Estimate transformations between the global common space and each subject’s raw response data.
3. Apply searchlights again:
   - Extract local representations from the global common space.
   - Extract corresponding local response data from each subject.
4. Estimate searchlight-level transformation matrices and **combine them into a global transformation**.



#### 2) Searchlight-only workflow (`xform_sls_ucon*`)

The functions **`xform_sls_ucon`** and **`xform_sls_ucon_mega`** **skip** the construction of a global common space.

**Workflow:**
1. Estimate searchlight-level common spaces directly.
2. Compute subject-specific transformations **within each searchlight**.
3. **Combine** local transformation matrices, without ever forming a global common space.



#### In short

- **`xform_sls_con*`**: *global common space → searchlight transformations → global transformation*  
- **`xform_sls_ucon*`**: *searchlight common spaces → searchlight transformations → combined transformation*


#### Concatenated workflow (global common space required)

In [ ]:
# --------------------------------------------------
# Estimate subject-to-common-space transformation matrices
# (Global common space–based workflow)
#
# The functions xform_sls_con / xform_sls_con_mega estimate
# transformation matrices from each subject’s response data
# to a precomputed global common space.
#
# The usage is similar to cspace_sls:
# - You need to specify where the subject (source) data are
# - And which common space (target) to use
# --------------------------------------------------


# --------------------------------------------------
# Surface data
# --------------------------------------------------
ha.xform_sls_con(
    work_dir,
    sls=sls_surf,                       # searchlight list (surface)
    dists=dists_surf,                   # corresponding distance list
    radius=20,                          # searchlight radius
    sub_list=sub_list,                  # subject list

    # Source data location
    source_step='resample',             # preprocessing step of subject data
    modality='response',                # modality folder
    structure_name=['hemi-L', 'hemi-R'],# surface structures

    # Target common space
    task_name='rhaPRs',                 # common space name
    cspace_desc={'alg': 'procrustes'},  # descriptor to select the correct common space

    # Procrustes options
    reflection=True,                    # allow reflection (recommended)
    scaling=False,                      # disable scaling (recommended)

    # Searchlight combination
    weight=True,                        # distance-weighted combination
    chunk_size=2000,                    # number of searchlights processed per chunk

    # Miscellaneous settings
    dtype=np.float32,
    njobs=5,
    scope='sls',                        # used for output naming
    json_path=json_path,
    overwrite=False,
    log_num='raiders_01',
    run='01'                            # file filter
)


# --------------------------------------------------
# Volume data
#
# In addition to xform_sls_con, you can also use
# xform_sls_con_mega, which has the same interface
# but is optimized for larger subject samples
# (e.g. > ~20 subjects).
# --------------------------------------------------
ha.xform_sls_con_mega(
    work_dir,
    sls=sls_vol,                        # searchlight list (volume)
    dists=dists_vol,                    # corresponding distance list
    radius=4,                           # searchlight radius
    sub_list=sub_list,

    # Source data location
    source_step='resample',
    modality='response',
    structure_name=[
        'volume-subcortical-L',
        'volume-subcortical-R',
        'volume-subcortical-BRAIN_STEM'
    ],

    # Target common space
    task_name='rhaPRv',
    cspace_desc={'alg': 'procrustes'},

    # Procrustes options
    reflection=True,
    scaling=False,

    # Searchlight combination
    weight=True,
    chunk_size=2000,

    # Miscellaneous settings
    dtype=np.float32,
    njobs=5,
    scope='sls',
    json_path=json_path,
    overwrite=False,
    log_num='raiders_01',
    run='01'
)


#### Un-concatenated workflow (searchlight common space required)

In [ ]:
# --------------------------------------------------
# High-level RHA functions
# `xform_sls_ucon` and `xform_sls_ucon_mega` integrate
# common space construction and transformation matrix
# estimation into a single workflow.
# --------------------------------------------------

# -------------------------
# Surface-based RHA
# -------------------------
ha.xform_sls_ucon(
        work_dir,
        sls=sls_surf,
        dists=dists_surf,
        radius=20,
        sub_list=sub_list,
        source_step='resample',
        modality='response',
        structure_name=['hemi-L', 'hemi-R'],
        task_name='rhaPRs',         # 'ucon' will be appended automatically
        cspace_kind='procrustes',
        common_topography=False,
        demean=True,
        start_sub=0,
        save_cspace=False,
        reflection=True,
        scaling=False,
        xform_weight=True,
        chunk_size=2000,
        dtype=np.float32,
        njobs=5,
        scope='sls',
        json_path=json_path,
        overwrite=False,
        log_num='raiders_01',
        verbose=0,
        run='01',
)

# -------------------------
# Volume-based RHA
# -------------------------
ha.xform_sls_ucon_mega(
        work_dir,
        sls=sls_vol,
        dists=dists_vol,
        radius=4,
        sub_list=sub_list,
        source_step='resample',
        modality='response',
        structure_name=[
            'volume-subcortical-L',
            'volume-subcortical-R',
            'volume-subcortical-BRAIN_STEM'
        ],
        task_name='rhaPCAv',
        cspace_kind='pca',
        common_topography=True,
        demean=True,
        start_sub=0,
        save_cspace=False,          # saving searchlight common space (False is recommended)
        reflection=True,
        scaling=False,
        xform_weight=True,
        chunk_size=2000,
        dtype=np.float32,
        njobs=40,
        scope='sls',
        json_path=json_path,
        overwrite=False,
        log_num='raiders_01',
        verbose=0,
        run='01',
)


### 2.3 Applying Transformation Matrices

This step is relatively straightforward. Once the transformation matrices have been estimated, the aligned data can be obtained by multiplying the original response data by the corresponding transformation matrix (i.e., \( A @ R \)). For convenience and efficiency, we provide a dedicated function that applies these transformations in parallel to accelerate computation.


In [ ]:
# --------------------------------------------------
# Apply transformation matrices estimated from run 1
# to data from run 2
# --------------------------------------------------

# Surface data (left and right hemispheres)
for structure in ['hemi-L', 'hemi-R']:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step='resample',
        source_modality='response',
        source_structure_name=structure,
        source_name_filter={'run': '02'},     # target data to be aligned
        xform_modality='response',
        xform_structure_name=None,            # automatically match corresponding structure
        xform_name_filter={'task': 'rhaPRs'}, # locate transformation matrices
        njobs=2,
        verbose=1,
        dtype=np.float32,
        json_path=json_path,
        overwrite=False,
        log_num='raiders_01',
        suffix='run02'
    )

# Subcortical volume data
for structure in [
    'volume-subcortical-L',
    'volume-subcortical-R',
    'volume-subcortical-BRAIN_STEM'
]:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step='resample',
        source_modality='response',
        source_structure_name=structure,
        source_name_filter={'run': '02'},     # target data to be aligned
        xform_modality='response',
        xform_structure_name=None,            # automatically match corresponding structure
        xform_name_filter={'task': 'rhaPRv'}, # locate transformation matrices
        njobs=2,
        verbose=1,
        dtype=np.float32,
        json_path=json_path,
        overwrite=False,
        log_num='raiders_01',
        suffix='run02'
    )


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   3 out of   3 | elapsed:    5.0s finished



All alignment completed successfully!


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   3 out of   3 | elapsed:    2.1s finished



All alignment completed successfully!


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   3 out of   3 | elapsed:    5.5s finished


...

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   3 out of   3 | elapsed:    3.2s finished



All alignment completed successfully!


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.



All alignment completed successfully!


[Parallel(n_jobs=2)]: Done   3 out of   3 | elapsed:    0.9s finished


## 3. Recommended Pipelines and Parameters

Selecting appropriate parameters can be challenging, especially for new users. To simplify this process, we also provide a set of recommended pipeline scripts that allow you to perform the analysis with minimal configuration. In these scripts, all required inputs and parameters are defined at the beginning, making them easy to inspect and modify.

For full implementations and additional examples, please refer to the scripts located in `/HAkit/HAkit/pipeline`.
